# Lab 6-1 Create and Save DataFrames



In this lab you will learn:
- How to create Spark DataFrames from Python data structures
- How to specify a schema when creating DataFrames
- Differences between inferred and specified data types in Spark
- Loading data from external file sources into DataFrames
- Using Databricks utilities to inspect file contents
- Understanding the distinction between Python data and Spark DataFrames
- Clean up the lab volumes and tables


Data files used:

-  `sparksql_demodata.zip`: several small csv and json files of various formats

Lab volume

- catalog: `msbabigdata`
- schema: `spark`
- volume: `lab6_1`

## 1. Create DataFrame from Data

In [ ]:
data = [('Ankit',25,'M'),('Nancy',22,'F'),('saurabh',20,"M"),('Priyanka',26,'F')]

### 1.1 Create DataFrame from Python
We first create a Spark DataFrame `df` from a python list (of tuples) 

```python
df= spark.createDataFrame(python_list, [list_of_column_names])
```

In [ ]:
df= spark.createDataFrame(data, ['name', 'age', 'gender'])

- What data type did Spark infer? 
- How is `data` and `df` differ?

### 1.2 Specify the schema when you create DataFrame

One way of providing a schema is to provide a **DDL (data definition language) string**, `col1 type1, col2 type2, ...`, using

`createDataFrame(data,schema)`

where `schema` is the DDL string.

Use this approach to convert data into a Spark dataframe `df2`

Commonly used Spark Data Types include:
- `STRING`
- `INT`
- `BIGINT`
- `DOUBLE`
- `DECIMAL`
- `BOOLEAN`
- `DATE`
- `TIMESTAMP`

In [ ]:
df2 = spark.createDataFrame(data, "name string, age int, gender string")
df2.display()

name | age | gender
Ankit | 25 | M
Nancy | 22 | F
saurabh | 20 | M
Priyanka | 26 | F

## 2. Create DataFrame from a File Source

### 2.1 Preparation

First, prepare the volume to hold your data


In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS msbabigdata;
CREATE SCHEMA IF NOT EXISTS msbabigdata.spark;
CREATE VOLUME IF NOT EXISTS msbabigdata.spark.lab6_1;

Then, download and unzip [https://idsdl.csom.umn.edu/c/share/msba6330/sparksql_demodata.zip](https://idsdl.csom.umn.edu/c/share/msba6330/sparksql_demodata.zip), then upload the unzipped files to the `lab6_1` volume.

### 2.2 Create DataFrame From CSV

Run the following statement to obtain a path variable

In [ ]:
path = "/Volumes/msbabigdata/spark/lab6_1"

#### 2.2.1 First, view format of peopleheader.csv using Catalog Explorer, or the following command 

> `dbutils.fs.head('path')` is a DataBricks file system utility that is similar to Linux command `head` (but it displays first 64K of the file)

---
```python
print(dbutils.fs.head(path+'/peopleheader.csv'))
```

In [ ]:
print(dbutils.fs.head(path+"/peopleheader.csv"))

pcode,lastName,firstName,age
02134,Hopper,Grace,52
94020,Turing,Alan,32
94020,Lovelace,Ada,28
87501,Babbage,Charles,49
02134,Wirth,Niklaus,48


#### 2.2.2 Create DataFrame `peopleDF` from the CSV file

- Use Spark DataFrame reader `spark.read.csv(path [,options])`
- since the file has a header rule, we want the option of `header=True`
- we may let Spark infer schema from data using `inferSchema=True` 

In [ ]:
peopleDF = spark.read.csv(path+'/peopleheader.csv', header=True, inferSchema=True)
peopleDF.display()

pcode | lastName | firstName | age
2134 | Hopper | Grace | 52
94020 | Turing | Alan | 32
94020 | Lovelace | Ada | 28
87501 | Babbage | Charles | 49
2134 | Wirth | Niklaus | 48

### 2.3 Create DataFrame From Json

#### 2.3.1 View the `people.json` file using `dbutils.fs.head`

> this is sometimes called JSONL format, where each line is a json document.


In [ ]:
print(dbutils.fs.head(path+"/people.json"))


{"name":"Alice", "pcode":"94304"}
{"name":"Brayden", "age":30, "pcode":"94304"}
{"name":"Carla", "age":19, "pcode":"10036"}
{"name":"Diana", "age":46}
{"name":"Étienne", "pcode":"94104"}


#### 2.3.2 Use the file to create `peopleJsonDF`

- Use `spark.read.json(path)`

In [ ]:
peopleJsonDF = spark.read.json(path+"/people.json")
peopleJsonDF.display()


age | name | pcode
None | Alice | 94304
30 | Brayden | 94304
19 | Carla | 10036
46 | Diana | None
None | Étienne | 94104

## 3. Save DataFrame

We often needs to save intermediate or final results (DataFrames) to disk for longer-term use.

### 3.1 Save DF to Unity Catalog volume

#### 3.1.1 Save `peopleDF` to the same volume's `people` folder. 

- Use `[DF].write.save(path)`
- We commonly use `.mode("overwrite")` to avoid "file already exists" errors.

> By default it saves a Delta Lake table

In [ ]:
peopleDF.write.mode("overwrite").save(path+"/people")

#choose mode overwrite bcoz we want indempotent run - if you run the same work twice you see the same result.


#### 3.1.2 Then use `dbutils.fs.ls('path')` to list the resulting files.

> `dbutils.fs.ls` is another Databricks file system utility, which is simiar to Linux's `ls` command.

> You can use `display(dbutils.fs.ls('path'))` to render the result in a more user-friendly manner

- Did Spark save a single file?
- What format was it saving in?

In [ ]:
display(dbutils.fs.ls(path + '/people'))

path | name | size | modificationTime
dbfs:/Volumes/msbabigdata/spark/lab6_1/people/_delta_log/ | _delta_log/ | 0 | 1771275580497
dbfs:/Volumes/msbabigdata/spark/lab6_1/people/part-00000-74cac315-0630-44f2-8f16-c7adbff597b6.c000.snappy.parquet | part-00000-74cac315-0630-44f2-8f16-c7adbff597b6.c000.snappy.parquet | 1346 | 1771275419000

### 3.2 Save DF to Unity Catalog Table

When your data is structured enough, you may be ready to convert it to a Spark table (so you can use SQL to query it)

In [ ]:
peopleDF.write.mode('overwrite').saveAsTable("default.people")


#### 3.2.1 Save peopleDF to a table named `people` in the `default` database
- Save to a table using `[DF].write.saveAsTable("schema.table")`
- Again, `overwrite` mode is common

#### 3.2.2 Display the first 10 rows from the table using Spark SQL

In [ ]:
%sql
select* from default.people limit 10;

pcode | lastName | firstName | age
2134 | Hopper | Grace | 52
94020 | Turing | Alan | 32
94020 | Lovelace | Ada | 28
87501 | Babbage | Charles | 49
2134 | Wirth | Niklaus | 48

#### 3.2.3 Read the table as a DataFrame 

- using `spark.table("schema.table")`

In [ ]:
peopleDF2 = spark.read.table("default.people")
peopleDF2.display()

pcode | lastName | firstName | age
2134 | Hopper | Grace | 52
94020 | Turing | Alan | 32
94020 | Lovelace | Ada | 28
87501 | Babbage | Charles | 49
2134 | Wirth | Niklaus | 48

### 3.3 Save DF as Temp View

Sometimes, you are not yet to create a table, but you still want to be able to query it using SQL, you may create a [Temp] View instead.

#### 3.3.1 Save `peopleJsonDF` as a temporary view `v_people`

- using `createOrReplaceTempView(view_name)`

In [ ]:
peopleJsonDF.createOrReplaceTempView("v_people")

####  3.3.2 Then query the first 10 rows using Spark SQL

- Would be the temp table `v_people` accessible in another notebook?
- When you close this notebook and reopen it, will `v_people` still be there?

No is the answer to both the questions above

In [ ]:
%sql
select*from v_people LIMIT 10;

age | name | pcode
None | Alice | 94304
30 | Brayden | 94304
19 | Carla | 10036
46 | Diana | None
None | Étienne | 94104

## 4. Clean up

### 4.1 Drop the volume `lab6_1`

- using `drop volume if exists [catalog].[schema].[volume]`

In [ ]:
%sql
drop volume if exists msbabigdata.spark.lab6_1;

### 4.2 Drop the table `people` from the default database


In [ ]:
%sql
Drop table if exists default.people;

This is the end of the Lab